# Extraer datos de las escuelas en Morelia
Esto es para mi papá

In [1]:
import os
import sys
import time
import dotenv
import warnings
from dataclasses import dataclass, asdict

sys.path.insert(1, "../")
warnings.filterwarnings("ignore")
dotenv.load_dotenv("/Users/mariano/.keys/.env")
sys.path.insert(2, "../../../../../jobs/prima/sdr-agent/")

PATH_DATA = '../../data/'
CITY_NAME = 'MORELIA'
CRS = 4326

In [2]:
from places_api import PlacesAPI, Location
from api.firebase_api import FirebaseAPI
from typing import List, Optional, Dict
from pprint import pprint
import geopandas as gpd
import pandas as pd
import h3pandas

In [20]:
city = (
    gpd
    .read_file(os.path.join(PATH_DATA, "localidadurbana", "LocalidadUrbana.shp"))
    .to_crs(CRS)
    .pipe(lambda df: df[df.NOM == CITY_NAME])
    .h3.polyfill_resample(9)
    # mide 460 metros de largo
    # así que tengo que usar radios de 230
)

print(f"Found {city.shape[0]} hexagons")

firestore = FirebaseAPI(
    service_account_path="/Users/mariano/.keys/cognitia-firestore-service-account.json",
    firestore_database_id="google-places"
)

places = PlacesAPI(api_key=os.getenv("GOOGLE_MAPS_API_KEY"), timeout=5)

Found 461 hexagons


In [21]:
@dataclass
class PlaceDetails:
    id:str
    address:str
    latitude:float
    longitude:float
    google_maps_uri:str
    total_ratings:int
    total_photos:int
    region_code:str
    postal_code:str
    state:str
    city:str
    colony:str
    primary_type:str
    rating:Optional[float]=None
    website:Optional[str] = None
    name:Optional[str] = None
    types:Optional[List[str]] = None
    phone_number:Optional[str] = None
    business_status:Optional[str] = None    

    def to_dict(self) -> Dict:
        return asdict(self)

In [22]:
radius = 230
collection_name = "schools" 
search_types = ["secondary_school", "university", "school", "primary_school", "preschool"]
exclude_types = ["sports_club"]
centroides = city.centroid.values

for centroid in centroides:
    try:
        location = Location(
            latitude=centroid.y,
            longitude=centroid.x
        )

        results = places.nearby_search(
            location, 
            radius_meters=radius, 
            included_types=search_types, 
            excluded_types=exclude_types
        )

        for place in results:
            try:
                time.sleep(.1)
                details = places.place_details(place_id=place)

                # Safe access to postalAddress
                postal_address = details.get('postalAddress') or {}

                # Damos formato al place details
                place_details = PlaceDetails(
                    id=place.id,
                    address=place.formatted_address,
                    latitude=place.location.latitude,
                    longitude=place.location.longitude,
                    google_maps_uri=details.get('googleMapsUri'),
                    total_ratings=place.user_rating_count,
                    total_photos=len(details.get('photos')) if details.get('photos') else 0,
                    region_code=postal_address.get('regionCode'),
                    postal_code=postal_address.get('postalCode'),
                    state=postal_address.get('administrativeArea'),
                    city=postal_address.get('locality'),
                    colony=postal_address.get('sublocality'),
                    primary_type=place.primary_type,
                    rating=place.rating,
                    website=details.get('websiteUri'),
                    name=details.get('displayName', {}).get('text') if details.get('displayName') else None,
                    types=place.types,
                    phone_number=details.get('internationalPhoneNumber', "").replace(" ", "") if details.get('internationalPhoneNumber') else "",
                    business_status=details.get('businessStatus')
                )

                firestore.firestore_set_document(
                    collection=collection_name,
                    document_id=place_details.id,
                    data=place_details.to_dict()
                )

                print(place_details.name, end='\r')
            except Exception as e:
                print(f"\nError processing place {place.id if hasattr(place, 'id') else 'unknown'}: {str(e)}")
                continue

    except Exception as e:
        print(f"\nError processing centroid at ({centroid.y}, {centroid.x}): {str(e)}")
        continue

Universidad Virtual y a Distancia de las Américasorres Manzoelia Las Américas de Hidalgos de Hidalgo

In [27]:
total_results = firestore.firestore_query_collection(collection=collection_name, limit=1000)

In [29]:
results = firestore.firestore_query_collection(
    collection=collection_name,
    limit=1_500
)

In [37]:
(
    pd
    .DataFrame(results)
    .query('business_status == "OPERATIONAL"')
    [[
        "name", "primary_type", "phone_number", "website", 
        "address", "google_maps_uri", "colony", "total_photos", 
        "total_ratings", "rating", "types", 
        "latitude", "longitude"
    ]]
    .to_clipboard(index=False)
)